
# CAPFG 10 — Frozen Confirmatory Evaluation

## Calibration-Aware Probabilistic Feature Generation for Data-Efficient Plant Disease Classification

This notebook performs the **frozen confirmatory evaluation** of CA-PFG.

### Frozen main method

\[
\text{CA-PFG}
=
\text{Support-only probabilistic feature generation}
+
\text{Normalized synthetic influence budget}
+
\text{Calibration-aware multi-objective selection}.
\]

The main method does **not** use:

- GAT;
- GCN;
- anisotropic covariance shaping;
- class-hardness covariance scaling;
- sample-level reliability weighting.

Those more complex mechanisms are retained only as controlled ablations.

### Confirmatory protocol

- Datasets: PlantDoc and PlantVillage
- \(K \in \{5, 10, 20\}\)
- 10 fresh confirmatory seeds
- 60 few-shot tasks in total
- frozen DINOv2-Small cached features
- support-only PCA and probabilistic estimation
- validation used only for model selection
- test evaluated only after selection
- frozen config hash attached to every task

Run the technical smoke task first. Only after the protocol audit passes should the full 60-task run be enabled.


In [ ]:

# Cell 2 — Imports and environment
from __future__ import annotations

import json
import math
import os
import sys
import hashlib
import shutil
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import scipy
from scipy.special import logsumexp
from scipy.stats import chi2
from scipy.linalg import solve_triangular
from sklearn import __version__ as sklearn_version
from sklearn.covariance import OAS
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
)
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn_version)


In [ ]:

# Cell 3 — Frozen confirmatory configuration

@dataclass(frozen=True)
class CAPFGConfig:
    experiment_version: str = "CAPFG10_frozen_confirmatory"

    # Confirmatory task grid
    confirmatory_datasets: Tuple[str, ...] = ("plantdoc", "plantvillage")
    confirmatory_k_values: Tuple[int, ...] = (5, 10, 20)
    confirmatory_seeds: Tuple[int, ...] = (
        137, 139, 149, 151, 157,
        163, 167, 173, 179, 181,
    )

    # Technical smoke
    smoke_dataset: str = "plantdoc"
    smoke_k: int = 5
    smoke_seed: int = 137

    # Frozen representation
    backbone_name: str = "facebook/dinov2-small"
    feature_dimension: int = 384
    pca_dimension: int = 64
    pca_whiten: bool = False

    # Frozen probabilistic generator
    pooled_covariance_weight: float = 0.25
    gaussian_jitter: float = 1e-6
    synthetic_multiplier: int = 20
    use_antithetic_sampling: bool = True

    # Frozen classifier search space
    logreg_c_grid: Tuple[float, ...] = (10.0, 30.0, 100.0)
    synthetic_total_weight_grid: Tuple[float, ...] = (0.125, 0.25, 0.50, 1.00)
    max_iter_logreg: int = 5000

    # Frozen calibration-aware multi-objective selection
    f1_tolerance: float = 0.002
    nll_tolerance: float = 0.01
    ece_tolerance: float = 0.01
    ece_bins: int = 15

    # Controlled ablation: class-hardness scalar scaling
    fixed_min_covariance_scale: float = 0.65
    fixed_hardness_shrinkage: float = 1.00

    # Controlled ablation: reliability v2, frozen after Notebook 9 development
    temperature_grid: Tuple[float, ...] = (0.5, 1.0, 2.0, 5.0, 10.0)
    reliability_gamma_fixed: float = 0.5
    reliability_minimum: float = 0.05
    posterior_epsilon: float = 1e-12

    # Diagnostics
    ess_warning_threshold: float = 0.10
    top10_budget_warning_threshold: float = 0.70
    degeneracy_source_posterior_median: float = 0.995
    degeneracy_entropy_median: float = 0.005
    degeneracy_ess_ratio: float = 0.995
    minimum_reliability_p05_p95_span: float = 0.01
    numerical_tolerance: float = 1e-6

CONFIG = CAPFGConfig()

# First Kaggle run: keep False and run the smoke task only.
# After smoke overall_protocol_pass=True, set True and Run All.
RUN_FULL_CONFIRMATORY = False

# Set False only when you explicitly want a minimal run without controlled ablations.
INCLUDE_CONTROLLED_ABLATIONS = True

# Optional path overrides. Leave None for recursive auto-discovery.
MANUAL_PATHS = {
    "manifest_split": None,
    "plantdoc_features": None,
    "plantdoc_metadata": None,
    "plantvillage_features": None,
    "plantvillage_metadata": None,
}

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/mnt/data"),
    Path.cwd(),
]

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/capfg10_outputs")
else:
    OUTPUT_DIR = Path.cwd() / "capfg10_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "synthetic_diagnostics").mkdir(parents=True, exist_ok=True)

print(CONFIG)
print("Output directory:", OUTPUT_DIR)
print("RUN_FULL_CONFIRMATORY:", RUN_FULL_CONFIRMATORY)
print("INCLUDE_CONTROLLED_ABLATIONS:", INCLUDE_CONTROLLED_ABLATIONS)


In [ ]:

# Cell 4 — Reproducibility and frozen config serialization

def stable_int_hash(value: Any, modulo: int = 2**32 - 1) -> int:
    digest = hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % modulo


def derive_seed(base_seed: int, component_name: str) -> int:
    return stable_int_hash(f"{int(base_seed)}::{component_name}")


def config_to_jsonable(config: CAPFGConfig) -> Dict[str, Any]:
    value = asdict(config)
    return json.loads(json.dumps(value, default=str))


def stable_hash_config(config: CAPFGConfig) -> str:
    payload = json.dumps(config_to_jsonable(config), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def write_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False, default=str)


CONFIG_HASH = stable_hash_config(CONFIG)
write_json(config_to_jsonable(CONFIG), OUTPUT_DIR / "development_config.json")
(OUTPUT_DIR / "development_config_hash.txt").write_text(CONFIG_HASH + "\n", encoding="utf-8")

print("Configuration SHA-256:", CONFIG_HASH)

In [ ]:

# Cell 5 — File discovery and schema harmonization

EXPECTED_FILENAMES = {
    "manifest_split": "manifest_split.csv",
    "plantdoc_features": "plantdoc_features.npy",
    "plantdoc_metadata": "plantdoc_metadata.csv",
    "plantvillage_features": "plantvillage_features.npy",
    "plantvillage_metadata": "plantvillage_metadata.csv",
}


def resolve_file(key: str) -> Optional[Path]:
    manual = MANUAL_PATHS.get(key)
    if manual:
        path = Path(manual)
        if not path.exists():
            raise FileNotFoundError(f"Manual path for {key} does not exist: {path}")
        return path

    filename = EXPECTED_FILENAMES[key]
    candidates: List[Path] = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.rglob(filename))

    # Remove output copies and duplicates.
    unique = []
    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if OUTPUT_DIR.resolve() in resolved.parents:
            continue
        unique.append(path)

    if not unique:
        return None

    unique.sort(key=lambda p: (len(p.parts), len(str(p))))
    if len(unique) > 1:
        print(f"[WARN] Multiple matches for {filename}; using {unique[0]}")
        for extra in unique[1:4]:
            print("       alternative:", extra)
    return unique[0]


def pick_column(frame: pd.DataFrame, candidates: Sequence[str], required: bool = False) -> Optional[str]:
    lower_map = {str(col).strip().lower(): col for col in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise KeyError(f"None of the required columns found: {candidates}. Available: {list(frame.columns)}")
    return None


def normalize_dataset_name(value: Any) -> str:
    text = str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")
    if "plantdoc" in text:
        return "plantdoc"
    if "plantvillage" in text:
        return "plantvillage"
    return text


def normalize_split_name(value: Any) -> str:
    text = str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")
    if text in {"train", "training", "tr"}:
        return "train"
    if text in {"val", "valid", "validation", "dev"}:
        return "validation"
    if text in {"test", "testing", "te"}:
        return "test"
    return text


ID_CANDIDATES = ("image_id", "sample_id", "id", "uid")
PATH_CANDIDATES = (
    "relative_path", "rel_path", "image_path", "filepath",
    "file_path", "path", "filename", "file_name"
)
LABEL_CANDIDATES = ("label_idx", "class_idx", "class_id", "target", "y", "label")
CLASS_NAME_CANDIDATES = ("class_name", "label_name", "category", "class")
SPLIT_CANDIDATES = ("split", "subset", "partition")
DATASET_CANDIDATES = ("dataset", "dataset_name")


def create_join_key(frame: pd.DataFrame) -> Tuple[Optional[pd.Series], Optional[str]]:
    id_col = pick_column(frame, ID_CANDIDATES)
    if id_col is not None:
        return frame[id_col].astype(str).str.strip(), f"id:{id_col}"

    path_col = pick_column(frame, PATH_CANDIDATES)
    if path_col is not None:
        key = frame[path_col].astype(str).str.replace("\\\\", "/", regex=False).str.strip()
        return key, f"path:{path_col}"

    return None, None


def harmonize_metadata(
    metadata: pd.DataFrame,
    manifest: Optional[pd.DataFrame],
    dataset_key: str,
) -> pd.DataFrame:
    meta = metadata.copy().reset_index(drop=True)
    meta["feature_index"] = np.arange(len(meta), dtype=int)

    # Filter manifest by dataset when possible.
    man = None
    if manifest is not None:
        man = manifest.copy()
        dataset_col = pick_column(man, DATASET_CANDIDATES)
        if dataset_col is not None:
            normalized = man[dataset_col].map(normalize_dataset_name)
            man = man.loc[normalized == dataset_key].copy()

    meta_split_col = pick_column(meta, SPLIT_CANDIDATES)
    meta_label_col = pick_column(meta, LABEL_CANDIDATES)
    meta_class_col = pick_column(meta, CLASS_NAME_CANDIDATES)

    needs_manifest = meta_split_col is None or (meta_label_col is None and meta_class_col is None)

    if needs_manifest:
        if man is None or len(man) == 0:
            raise ValueError(
                "Metadata does not contain enough split/label information, and no compatible "
                "manifest_split.csv was found."
            )

        meta_key, meta_key_desc = create_join_key(meta)
        man_key, man_key_desc = create_join_key(man)
        if meta_key is None or man_key is None:
            raise ValueError(
                "Cannot merge feature metadata with manifest. Add a shared image_id or path column, "
                "or set MANUAL_PATHS to the correct files."
            )

        meta = meta.copy()
        man = man.copy()
        meta["_join_key"] = meta_key
        man["_join_key"] = man_key
        if meta["_join_key"].duplicated().any():
            raise ValueError(f"Duplicate metadata join keys using {meta_key_desc}.")
        if man["_join_key"].duplicated().any():
            # Exact duplicate manifest rows are harmless; ambiguous keys are not.
            duplicate_groups = man.loc[man["_join_key"].duplicated(keep=False), "_join_key"]
            raise ValueError(
                f"Duplicate manifest join keys using {man_key_desc}; examples: "
                f"{duplicate_groups.head(5).tolist()}"
            )

        keep_cols = ["_join_key"]
        for candidates in (SPLIT_CANDIDATES, LABEL_CANDIDATES, CLASS_NAME_CANDIDATES, DATASET_CANDIDATES):
            col = pick_column(man, candidates)
            if col is not None and col not in keep_cols:
                keep_cols.append(col)

        meta = meta.merge(man[keep_cols], on="_join_key", how="left", suffixes=("", "_manifest"))
        if meta.isna().all(axis=1).any():
            raise ValueError("Unexpected fully empty rows after metadata/manifest merge.")

    split_col = pick_column(meta, SPLIT_CANDIDATES)
    if split_col is None:
        split_col = pick_column(meta, tuple(f"{c}_manifest" for c in SPLIT_CANDIDATES))
    if split_col is None:
        raise KeyError("No split column found after harmonization.")

    label_col = pick_column(meta, LABEL_CANDIDATES)
    if label_col is None:
        label_col = pick_column(meta, tuple(f"{c}_manifest" for c in LABEL_CANDIDATES))

    class_col = pick_column(meta, CLASS_NAME_CANDIDATES)
    if class_col is None:
        class_col = pick_column(meta, tuple(f"{c}_manifest" for c in CLASS_NAME_CANDIDATES))

    meta["split_canonical"] = meta[split_col].map(normalize_split_name)

    if label_col is not None and pd.api.types.is_numeric_dtype(meta[label_col]):
        meta["label_canonical"] = pd.to_numeric(meta[label_col], errors="raise").astype(int)
    elif label_col is not None:
        numeric = pd.to_numeric(meta[label_col], errors="coerce")
        if numeric.notna().all():
            meta["label_canonical"] = numeric.astype(int)
        else:
            labels_as_text = meta[label_col].astype(str)
            mapping = {name: idx for idx, name in enumerate(sorted(labels_as_text.unique()))}
            meta["label_canonical"] = labels_as_text.map(mapping).astype(int)
    elif class_col is not None:
        classes_as_text = meta[class_col].astype(str)
        mapping = {name: idx for idx, name in enumerate(sorted(classes_as_text.unique()))}
        meta["label_canonical"] = classes_as_text.map(mapping).astype(int)
    else:
        raise KeyError("No label or class-name column found after harmonization.")

    if class_col is not None:
        meta["class_name_canonical"] = meta[class_col].astype(str)
    else:
        meta["class_name_canonical"] = meta["label_canonical"].astype(str)

    join_key, _ = create_join_key(meta)
    if join_key is None:
        meta["sample_key"] = dataset_key + "::" + meta["feature_index"].astype(str)
    else:
        meta["sample_key"] = dataset_key + "::" + join_key.astype(str)

    valid_splits = {"train", "validation", "test"}
    unknown = sorted(set(meta["split_canonical"]) - valid_splits)
    if unknown:
        raise ValueError(f"Unknown split values after normalization: {unknown}")

    if meta["sample_key"].duplicated().any():
        raise ValueError("Duplicate sample keys detected.")

    return meta

In [ ]:

# Cell 6 — Load cached feature bundles

@dataclass
class DatasetBundle:
    dataset: str
    features: np.ndarray
    metadata: pd.DataFrame
    class_labels: np.ndarray
    paths: Dict[str, Optional[str]]


def load_dataset_bundle(dataset_key: str) -> DatasetBundle:
    dataset_key = normalize_dataset_name(dataset_key)
    feature_key = f"{dataset_key}_features"
    metadata_key = f"{dataset_key}_metadata"

    feature_path = resolve_file(feature_key)
    metadata_path = resolve_file(metadata_key)
    manifest_path = resolve_file("manifest_split")

    if feature_path is None:
        raise FileNotFoundError(
            f"Could not find {EXPECTED_FILENAMES[feature_key]}. "
            "Set MANUAL_PATHS or attach the Notebook 2 output as a Kaggle Input."
        )
    if metadata_path is None:
        raise FileNotFoundError(
            f"Could not find {EXPECTED_FILENAMES[metadata_key]}. "
            "Set MANUAL_PATHS or attach the Notebook 2 output as a Kaggle Input."
        )

    features = np.load(feature_path, mmap_mode=None)
    metadata = pd.read_csv(metadata_path)
    manifest = pd.read_csv(manifest_path) if manifest_path is not None else None

    if features.ndim != 2:
        raise ValueError(f"Expected a 2-D feature matrix, got {features.shape}")
    if len(features) != len(metadata):
        raise ValueError(
            f"Feature/metadata row mismatch: {len(features)} vs {len(metadata)}"
        )
    if not np.isfinite(features).all():
        raise ValueError("Feature matrix contains NaN or infinite values.")

    harmonized = harmonize_metadata(metadata, manifest, dataset_key)
    class_labels = np.sort(harmonized["label_canonical"].unique())

    split_counts = harmonized["split_canonical"].value_counts().to_dict()
    class_counts = (
        harmonized.groupby(["split_canonical", "label_canonical"])
        .size()
        .groupby(level=0)
        .agg(["min", "max"])
    )

    print(f"\nLoaded {dataset_key}:")
    print("  features:", feature_path)
    print("  metadata:", metadata_path)
    print("  manifest:", manifest_path)
    print("  feature shape:", features.shape)
    print("  split counts:", split_counts)
    print("  classes:", len(class_labels))
    display(class_counts)

    return DatasetBundle(
        dataset=dataset_key,
        features=np.asarray(features, dtype=np.float32),
        metadata=harmonized,
        class_labels=class_labels,
        paths={
            "features": str(feature_path),
            "metadata": str(metadata_path),
            "manifest": str(manifest_path) if manifest_path is not None else None,
        },
    )


BUNDLES: Dict[str, DatasetBundle] = {}
for dataset_name in CONFIG.confirmatory_datasets:
    BUNDLES[dataset_name] = load_dataset_bundle(dataset_name)

In [ ]:

# Cell 7 — Task split construction and support-only preprocessing

@dataclass
class TaskSplit:
    support_indices: np.ndarray
    validation_indices: np.ndarray
    test_indices: np.ndarray
    class_labels: np.ndarray


@dataclass
class SupportPreprocessor:
    pca: PCA
    n_components: int


def build_task_split(bundle: DatasetBundle, k: int, seed: int) -> TaskSplit:
    meta = bundle.metadata
    train = meta.loc[meta["split_canonical"] == "train"]
    validation_indices = meta.index[meta["split_canonical"] == "validation"].to_numpy(dtype=int)
    test_indices = meta.index[meta["split_canonical"] == "test"].to_numpy(dtype=int)

    support_parts = []
    for label in bundle.class_labels:
        candidates = train.index[train["label_canonical"] == label].to_numpy(dtype=int)
        if len(candidates) < k:
            raise ValueError(
                f"{bundle.dataset}: class {label} has only {len(candidates)} train samples, K={k}."
            )
        rng = np.random.default_rng(derive_seed(seed, f"support::{bundle.dataset}::{k}::{label}"))
        chosen = np.sort(rng.choice(candidates, size=k, replace=False))
        support_parts.append(chosen)

    support_indices = np.sort(np.concatenate(support_parts))

    support_set = set(support_indices.tolist())
    validation_set = set(validation_indices.tolist())
    test_set = set(test_indices.tolist())

    if support_set & validation_set:
        raise AssertionError("Support/validation overlap detected.")
    if support_set & test_set:
        raise AssertionError("Support/test overlap detected.")
    if validation_set & test_set:
        raise AssertionError("Validation/test overlap detected.")

    support_labels = meta.loc[support_indices, "label_canonical"].to_numpy(dtype=int)
    observed, counts = np.unique(support_labels, return_counts=True)
    if not np.array_equal(observed, bundle.class_labels):
        raise AssertionError("Support does not cover all classes.")
    if not np.all(counts == k):
        raise AssertionError(f"Support is not exactly {k}-shot per class: {dict(zip(observed, counts))}")

    return TaskSplit(
        support_indices=support_indices,
        validation_indices=validation_indices,
        test_indices=test_indices,
        class_labels=bundle.class_labels.copy(),
    )


def fit_support_preprocessor(
    X_support_raw: np.ndarray,
    config: CAPFGConfig,
    seed: int,
) -> SupportPreprocessor:
    X_support_l2 = normalize(X_support_raw, norm="l2")
    n_components = min(
        config.pca_dimension,
        X_support_l2.shape[0] - 1,
        X_support_l2.shape[1],
    )
    if n_components < 2:
        raise ValueError(f"PCA dimension too small: {n_components}")

    pca = PCA(
        n_components=n_components,
        whiten=config.pca_whiten,
        svd_solver="full",
        random_state=seed,
    )
    pca.fit(X_support_l2)
    return SupportPreprocessor(pca=pca, n_components=n_components)


def transform_with_support_preprocessor(
    preprocessor: SupportPreprocessor,
    X_raw: np.ndarray,
) -> np.ndarray:
    X_l2 = normalize(X_raw, norm="l2")
    X_pca = preprocessor.pca.transform(X_l2)
    if not np.isfinite(X_pca).all():
        raise ValueError("Non-finite transformed features.")
    return np.asarray(X_pca, dtype=np.float64)

In [ ]:

# Cell 8 — Stable Gaussian estimation

@dataclass
class GaussianModel:
    class_labels: np.ndarray
    means: np.ndarray
    class_covariances: np.ndarray
    base_covariances: np.ndarray
    pooled_covariance: np.ndarray
    oas_shrinkages: np.ndarray
    class_counts: np.ndarray


def ensure_positive_definite(
    covariance: np.ndarray,
    jitter: float,
) -> np.ndarray:
    covariance = np.asarray(covariance, dtype=np.float64)
    covariance = 0.5 * (covariance + covariance.T)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    floor = max(float(jitter), np.finfo(np.float64).eps)
    eigenvalues = np.maximum(eigenvalues, floor)
    repaired = (eigenvectors * eigenvalues) @ eigenvectors.T
    return 0.5 * (repaired + repaired.T)


def fit_oas_covariance(
    X: np.ndarray,
    jitter: float,
    assume_centered: bool = False,
) -> Tuple[np.ndarray, float]:
    estimator = OAS(assume_centered=assume_centered, store_precision=False)
    estimator.fit(np.asarray(X, dtype=np.float64))
    covariance = ensure_positive_definite(estimator.covariance_, jitter)
    return covariance, float(estimator.shrinkage_)


def estimate_gaussian_model(
    X_support: np.ndarray,
    y_support: np.ndarray,
    class_labels: np.ndarray,
    pooled_weight: float,
    jitter: float,
) -> GaussianModel:
    means = []
    class_covariances = []
    shrinkages = []
    counts = []
    residuals = []

    for label in class_labels:
        Xc = X_support[y_support == label]
        if len(Xc) < 2:
            raise ValueError(f"At least two support samples are required for class {label}.")
        mean_c = Xc.mean(axis=0)
        covariance_c, shrinkage_c = fit_oas_covariance(Xc, jitter, assume_centered=False)

        means.append(mean_c)
        class_covariances.append(covariance_c)
        shrinkages.append(shrinkage_c)
        counts.append(len(Xc))
        residuals.append(Xc - mean_c)

    residual_matrix = np.vstack(residuals)
    pooled_covariance, _ = fit_oas_covariance(
        residual_matrix,
        jitter,
        assume_centered=True,
    )

    base_covariances = []
    for covariance_c in class_covariances:
        base = (1.0 - pooled_weight) * covariance_c + pooled_weight * pooled_covariance
        base_covariances.append(ensure_positive_definite(base, jitter))

    return GaussianModel(
        class_labels=np.asarray(class_labels, dtype=int),
        means=np.asarray(means, dtype=np.float64),
        class_covariances=np.asarray(class_covariances, dtype=np.float64),
        base_covariances=np.asarray(base_covariances, dtype=np.float64),
        pooled_covariance=np.asarray(pooled_covariance, dtype=np.float64),
        oas_shrinkages=np.asarray(shrinkages, dtype=np.float64),
        class_counts=np.asarray(counts, dtype=int),
    )


def gaussian_log_likelihood(
    X: np.ndarray,
    means: np.ndarray,
    covariances: np.ndarray,
) -> np.ndarray:
    X = np.atleast_2d(np.asarray(X, dtype=np.float64))
    n_samples, dimension = X.shape
    n_classes = means.shape[0]
    output = np.empty((n_samples, n_classes), dtype=np.float64)
    constant = dimension * math.log(2.0 * math.pi)

    for class_position in range(n_classes):
        covariance = covariances[class_position]
        chol = np.linalg.cholesky(covariance)
        diff = X - means[class_position]
        solved = solve_triangular(chol, diff.T, lower=True, check_finite=False)
        mahalanobis = np.sum(solved * solved, axis=0)
        log_determinant = 2.0 * np.log(np.diag(chol)).sum()
        output[:, class_position] = -0.5 * (constant + log_determinant + mahalanobis)

    return output


def gaussian_posterior(
    X: np.ndarray,
    means: np.ndarray,
    covariances: np.ndarray,
) -> np.ndarray:
    log_likelihood = gaussian_log_likelihood(X, means, covariances)
    log_posterior = log_likelihood - logsumexp(log_likelihood, axis=1, keepdims=True)
    posterior = np.exp(log_posterior)
    posterior /= posterior.sum(axis=1, keepdims=True)
    return posterior

In [ ]:

# Cell 9 — Support-only leave-one-out class hardness for ablation

@dataclass
class HardnessResult:
    raw_hardness: np.ndarray
    final_hardness: np.ndarray
    true_class_loo_posterior: np.ndarray
    sample_labels: np.ndarray


def compute_leave_one_out_hardness(
    X_support: np.ndarray,
    y_support: np.ndarray,
    gaussian_model: GaussianModel,
    config: CAPFGConfig,
) -> HardnessResult:
    label_to_position = {
        int(label): position
        for position, label in enumerate(gaussian_model.class_labels)
    }

    true_posteriors = np.empty(len(X_support), dtype=np.float64)

    for sample_index in range(len(X_support)):
        label = int(y_support[sample_index])
        class_position = label_to_position[label]
        class_mask = y_support == label
        class_indices = np.flatnonzero(class_mask)
        retained_indices = class_indices[class_indices != sample_index]
        X_retained = X_support[retained_indices]

        if len(X_retained) < 2:
            raise ValueError(
                f"Leave-one-out estimation leaves fewer than two samples for class {label}."
            )

        loo_mean = X_retained.mean(axis=0)
        loo_covariance, _ = fit_oas_covariance(
            X_retained,
            config.gaussian_jitter,
            assume_centered=False,
        )
        loo_base_covariance = (
            (1.0 - config.pooled_covariance_weight) * loo_covariance
            + config.pooled_covariance_weight * gaussian_model.pooled_covariance
        )
        loo_base_covariance = ensure_positive_definite(
            loo_base_covariance,
            config.gaussian_jitter,
        )

        means = gaussian_model.means.copy()
        covariances = gaussian_model.base_covariances.copy()
        means[class_position] = loo_mean
        covariances[class_position] = loo_base_covariance

        posterior = gaussian_posterior(
            X_support[sample_index : sample_index + 1],
            means,
            covariances,
        )[0]
        true_posteriors[sample_index] = posterior[class_position]

    raw_hardness = []
    for label in gaussian_model.class_labels:
        class_values = true_posteriors[y_support == label]
        raw_hardness.append(1.0 - float(class_values.mean()))
    raw_hardness = np.clip(np.asarray(raw_hardness), 0.0, 1.0)

    global_hardness = float(raw_hardness.mean())
    shrinkage = config.fixed_hardness_shrinkage
    final_hardness = shrinkage * raw_hardness + (1.0 - shrinkage) * global_hardness
    final_hardness = np.clip(final_hardness, 0.0, 1.0)

    return HardnessResult(
        raw_hardness=raw_hardness,
        final_hardness=final_hardness,
        true_class_loo_posterior=true_posteriors,
        sample_labels=np.asarray(y_support, dtype=int),
    )


def covariance_scales_from_hardness(
    hardness: np.ndarray,
    minimum_scale: float,
) -> np.ndarray:
    scales = minimum_scale + (1.0 - minimum_scale) * (1.0 - hardness)
    return np.clip(scales, minimum_scale, 1.0)

In [ ]:

# Cell 10 — Antithetic probabilistic feature generation

@dataclass
class SyntheticBatch:
    features: np.ndarray
    labels: np.ndarray
    source_positions: np.ndarray
    pair_ids: np.ndarray
    geometry_name: str


def construct_generation_covariances(
    base_covariances: np.ndarray,
    scales: np.ndarray,
    jitter: float,
) -> np.ndarray:
    generated = []
    for covariance, scale in zip(base_covariances, scales):
        generated.append(ensure_positive_definite(float(scale) * covariance, jitter))
    return np.asarray(generated, dtype=np.float64)


def generate_antithetic_features(
    gaussian_model: GaussianModel,
    generation_covariances: np.ndarray,
    samples_per_class: int,
    task_seed: int,
    geometry_name: str,
    use_antithetic: bool = True,
) -> SyntheticBatch:
    all_features = []
    all_labels = []
    all_source_positions = []
    all_pair_ids = []
    next_pair_id = 0

    for class_position, label in enumerate(gaussian_model.class_labels):
        rng = np.random.default_rng(
            derive_seed(task_seed, f"synthetic_noise::{label}")
        )
        mean = gaussian_model.means[class_position]
        covariance = generation_covariances[class_position]
        chol = np.linalg.cholesky(covariance)
        dimension = mean.shape[0]

        if use_antithetic:
            n_pairs = samples_per_class // 2
            epsilon = rng.standard_normal((n_pairs, dimension))
            deviations = epsilon @ chol.T
            generated = np.vstack([mean + deviations, mean - deviations])
            pair_ids = np.concatenate([
                np.arange(next_pair_id, next_pair_id + n_pairs),
                np.arange(next_pair_id, next_pair_id + n_pairs),
            ])
            next_pair_id += n_pairs

            if len(generated) < samples_per_class:
                extra_epsilon = rng.standard_normal((1, dimension))
                extra = mean + extra_epsilon @ chol.T
                generated = np.vstack([generated, extra])
                pair_ids = np.concatenate([pair_ids, np.array([next_pair_id])])
                next_pair_id += 1
        else:
            epsilon = rng.standard_normal((samples_per_class, dimension))
            generated = mean + epsilon @ chol.T
            pair_ids = np.arange(next_pair_id, next_pair_id + samples_per_class)
            next_pair_id += samples_per_class

        generated = generated[:samples_per_class]
        pair_ids = pair_ids[:samples_per_class]

        all_features.append(generated)
        all_labels.append(np.full(samples_per_class, int(label), dtype=int))
        all_source_positions.append(
            np.full(samples_per_class, class_position, dtype=int)
        )
        all_pair_ids.append(pair_ids.astype(int))

    X_syn = np.vstack(all_features).astype(np.float64)
    y_syn = np.concatenate(all_labels)
    source_positions = np.concatenate(all_source_positions)
    pair_ids = np.concatenate(all_pair_ids)

    if not np.isfinite(X_syn).all():
        raise ValueError("Synthetic features contain NaN or infinite values.")

    return SyntheticBatch(
        features=X_syn,
        labels=y_syn,
        source_positions=source_positions,
        pair_ids=pair_ids,
        geometry_name=geometry_name,
    )

In [ ]:
# Cell 11 — Shared-metric reliability v2 evaluator for ablation

@dataclass
class TemperatureCalibrationResult:
    selected_temperature: float
    candidate_table: pd.DataFrame
    loo_squared_distances: np.ndarray
    loo_true_positions: np.ndarray


@dataclass
class ReliabilityResult:
    posterior: np.ndarray
    source_posterior: np.ndarray
    normalized_entropy: np.ndarray
    radial_centrality: np.ndarray
    raw_reliability: np.ndarray
    predicted_positions: np.ndarray
    shared_mahalanobis_to_source: np.ndarray
    generation_mahalanobis_to_source: np.ndarray
    selected_temperature: float


def shared_squared_mahalanobis(
    X: np.ndarray,
    means: np.ndarray,
    shared_covariance: np.ndarray,
) -> np.ndarray:
    X = np.atleast_2d(np.asarray(X, dtype=np.float64))
    chol = np.linalg.cholesky(shared_covariance)
    output = np.empty((len(X), len(means)), dtype=np.float64)
    for class_position, mean in enumerate(means):
        diff = X - mean
        solved = solve_triangular(chol, diff.T, lower=True, check_finite=False)
        output[:, class_position] = np.sum(solved * solved, axis=0)
    return output


def posterior_from_shared_distances(
    squared_distances: np.ndarray,
    temperature: float,
) -> np.ndarray:
    if temperature <= 0:
        raise ValueError("Temperature must be positive.")
    logits = -0.5 * np.asarray(squared_distances, dtype=np.float64) / float(temperature)
    logits -= logsumexp(logits, axis=1, keepdims=True)
    posterior = np.exp(logits)
    posterior /= posterior.sum(axis=1, keepdims=True)
    return posterior


def leave_one_out_shared_distances(
    X_support: np.ndarray,
    y_support: np.ndarray,
    class_labels: np.ndarray,
    full_means: np.ndarray,
    shared_covariance: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Leave each support sample out of its source-class prototype.
    The shared covariance is estimated once from support residuals only.
    This avoids self-confirmation through the source prototype while keeping
    temperature calibration computationally tractable.
    """
    label_to_position = {
        int(label): position for position, label in enumerate(class_labels)
    }
    all_distances = np.empty((len(X_support), len(class_labels)), dtype=np.float64)
    true_positions = np.empty(len(X_support), dtype=int)

    for held_out_index in range(len(X_support)):
        held_out_label = int(y_support[held_out_index])
        held_out_position = label_to_position[held_out_label]
        class_indices = np.flatnonzero(y_support == held_out_label)
        retained_class_indices = class_indices[class_indices != held_out_index]
        if len(retained_class_indices) < 2:
            raise ValueError(
                f"LOO prototype leaves fewer than two samples for class {held_out_label}."
            )

        means = full_means.copy()
        means[held_out_position] = X_support[retained_class_indices].mean(axis=0)

        all_distances[held_out_index] = shared_squared_mahalanobis(
            X_support[held_out_index : held_out_index + 1],
            means,
            shared_covariance,
        )[0]
        true_positions[held_out_index] = held_out_position

    return all_distances, true_positions

def calibrate_shared_temperature(
    X_support: np.ndarray,
    y_support: np.ndarray,
    gaussian_model: GaussianModel,
    config: CAPFGConfig,
) -> TemperatureCalibrationResult:
    loo_distances, true_positions = leave_one_out_shared_distances(
        X_support,
        y_support,
        gaussian_model.class_labels,
        gaussian_model.means,
        gaussian_model.pooled_covariance,
    )
    class_labels = gaussian_model.class_labels

    candidate_rows = []
    for temperature in config.temperature_grid:
        posterior = posterior_from_shared_distances(loo_distances, temperature)
        true_probability = np.clip(
            posterior[np.arange(len(posterior)), true_positions],
            config.posterior_epsilon,
            1.0,
        )
        nll = float(-np.log(true_probability).mean())
        predicted_positions = posterior.argmax(axis=1)
        accuracy = float((predicted_positions == true_positions).mean())
        entropy = -np.sum(
            np.clip(posterior, config.posterior_epsilon, 1.0)
            * np.log(np.clip(posterior, config.posterior_epsilon, 1.0)),
            axis=1,
        ) / math.log(len(class_labels))

        candidate_rows.append({
            "temperature": float(temperature),
            "loo_nll": nll,
            "loo_accuracy": accuracy,
            "loo_source_posterior_mean": float(true_probability.mean()),
            "loo_source_posterior_median": float(np.median(true_probability)),
            "loo_entropy_mean": float(entropy.mean()),
            "loo_entropy_median": float(np.median(entropy)),
        })

    candidate_table = pd.DataFrame(candidate_rows)
    best_nll = candidate_table["loo_nll"].min()
    selected_temperature = float(
        candidate_table.loc[
            np.isclose(candidate_table["loo_nll"], best_nll, atol=1e-12),
            "temperature",
        ].min()
    )

    return TemperatureCalibrationResult(
        selected_temperature=selected_temperature,
        candidate_table=candidate_table,
        loo_squared_distances=loo_distances,
        loo_true_positions=true_positions,
    )


def compute_source_mahalanobis(
    X: np.ndarray,
    source_positions: np.ndarray,
    means: np.ndarray,
    covariances: np.ndarray,
) -> np.ndarray:
    distances = np.empty(len(X), dtype=np.float64)
    for class_position in range(len(means)):
        mask = source_positions == class_position
        if not mask.any():
            continue
        chol = np.linalg.cholesky(covariances[class_position])
        diff = X[mask] - means[class_position]
        solved = solve_triangular(chol, diff.T, lower=True, check_finite=False)
        distances[mask] = np.sum(solved * solved, axis=0)
    return distances


def compute_synthetic_reliability_v2(
    synthetic: SyntheticBatch,
    gaussian_model: GaussianModel,
    generation_covariances: np.ndarray,
    temperature_calibration: TemperatureCalibrationResult,
    config: CAPFGConfig,
) -> ReliabilityResult:
    # Competition uses one common metric, not the class covariance used by the generator.
    shared_distances = shared_squared_mahalanobis(
        synthetic.features,
        gaussian_model.means,
        gaussian_model.pooled_covariance,
    )
    posterior = posterior_from_shared_distances(
        shared_distances,
        temperature_calibration.selected_temperature,
    )

    row_indices = np.arange(len(synthetic.features))
    source_posterior = posterior[row_indices, synthetic.source_positions]

    clipped = np.clip(posterior, config.posterior_epsilon, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = np.clip(
        entropy / math.log(posterior.shape[1]),
        0.0,
        1.0,
    )

    shared_mahalanobis_to_source = shared_distances[
        row_indices,
        synthetic.source_positions,
    ]
    generation_mahalanobis_to_source = compute_source_mahalanobis(
        synthetic.features,
        synthetic.source_positions,
        gaussian_model.means,
        generation_covariances,
    )

    # Chi-square survival is near 1 at the center and decreases in the tail.
    feature_dimension = synthetic.features.shape[1]
    radial_centrality = chi2.sf(
        generation_mahalanobis_to_source,
        df=feature_dimension,
    )
    radial_centrality = np.clip(radial_centrality, 0.0, 1.0)

    raw_reliability = (
        source_posterior
        * (1.0 - normalized_entropy)
        * np.sqrt(radial_centrality)
    )
    raw_reliability = np.clip(raw_reliability, 0.0, 1.0)

    return ReliabilityResult(
        posterior=posterior,
        source_posterior=source_posterior,
        normalized_entropy=normalized_entropy,
        radial_centrality=radial_centrality,
        raw_reliability=raw_reliability,
        predicted_positions=posterior.argmax(axis=1),
        shared_mahalanobis_to_source=shared_mahalanobis_to_source,
        generation_mahalanobis_to_source=generation_mahalanobis_to_source,
        selected_temperature=temperature_calibration.selected_temperature,
    )


def powered_reliability(
    raw_reliability: np.ndarray,
    gamma: float,
    minimum: float,
) -> np.ndarray:
    reliability = np.power(np.clip(raw_reliability, 0.0, 1.0), gamma)
    return np.clip(reliability, minimum, 1.0)


def normalize_synthetic_weights(
    reliability: np.ndarray,
    lambda_syn: float,
    n_real: int,
    tolerance: float,
) -> np.ndarray:
    """Global normalized budget; retained for uniform-weight baselines."""
    reliability = np.asarray(reliability, dtype=np.float64)
    if np.any(reliability < 0) or not np.isfinite(reliability).all():
        raise ValueError("Reliability must be finite and non-negative.")
    denominator = reliability.sum()
    if denominator <= 0:
        raise ValueError("Reliability sum is zero.")
    budget = float(lambda_syn) * int(n_real)
    weights = budget * reliability / denominator
    if abs(weights.sum() - budget) > tolerance:
        raise AssertionError(f"Synthetic budget mismatch: {weights.sum()} vs {budget}")
    return weights


def normalize_class_balanced_synthetic_weights(
    reliability: np.ndarray,
    source_positions: np.ndarray,
    lambda_syn: float,
    n_real: int,
    n_classes: int,
    tolerance: float,
) -> np.ndarray:
    """Equal class budgets; reliability redistributes weight within each source class."""
    reliability = np.asarray(reliability, dtype=np.float64)
    source_positions = np.asarray(source_positions, dtype=int)
    if len(reliability) != len(source_positions):
        raise ValueError("Reliability/source-position length mismatch.")
    if np.any(reliability < 0) or not np.isfinite(reliability).all():
        raise ValueError("Reliability must be finite and non-negative.")

    total_budget = float(lambda_syn) * int(n_real)
    class_budget = total_budget / int(n_classes)
    weights = np.zeros_like(reliability, dtype=np.float64)

    for class_position in range(n_classes):
        mask = source_positions == class_position
        if not mask.any():
            raise ValueError(f"No synthetic samples for source class position {class_position}.")
        class_reliability = reliability[mask]
        denominator = class_reliability.sum()
        if denominator <= 0:
            raise ValueError(f"Reliability sum is zero for class position {class_position}.")
        weights[mask] = class_budget * class_reliability / denominator

    if abs(weights.sum() - total_budget) > tolerance:
        raise AssertionError(
            f"Class-balanced synthetic budget mismatch: {weights.sum()} vs {total_budget}"
        )

    class_sums = np.array([
        weights[source_positions == class_position].sum()
        for class_position in range(n_classes)
    ])
    if not np.allclose(class_sums, class_budget, atol=tolerance, rtol=0.0):
        raise AssertionError("Class budgets are not equal within tolerance.")
    return weights


def effective_sample_size(weights: np.ndarray) -> float:
    weights = np.asarray(weights, dtype=np.float64)
    numerator = weights.sum() ** 2
    denominator = np.sum(weights ** 2)
    return float(numerator / denominator) if denominator > 0 else 0.0


def gini_coefficient(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 0:
        return float("nan")
    values = np.clip(values, 0.0, None)
    if np.allclose(values.sum(), 0.0):
        return 0.0
    sorted_values = np.sort(values)
    n = len(sorted_values)
    cumulative = np.cumsum(sorted_values)
    return float((n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n)


def top_budget_fraction(weights: np.ndarray, fraction: float) -> float:
    weights = np.asarray(weights, dtype=np.float64)
    n_top = max(1, int(math.ceil(len(weights) * fraction)))
    return float(np.sort(weights)[-n_top:].sum() / weights.sum())


def reliability_degeneracy_summary(
    reliability: ReliabilityResult,
    selected_weights: np.ndarray,
    config: CAPFGConfig,
) -> Dict[str, Any]:
    source_median = float(np.median(reliability.source_posterior))
    entropy_median = float(np.median(reliability.normalized_entropy))
    raw_p05, raw_p95 = np.quantile(reliability.raw_reliability, [0.05, 0.95])
    raw_span = float(raw_p95 - raw_p05)
    ess_ratio = float(effective_sample_size(selected_weights) / len(selected_weights))

    saturated_uniform = bool(
        source_median > config.degeneracy_source_posterior_median
        and entropy_median < config.degeneracy_entropy_median
        and ess_ratio > config.degeneracy_ess_ratio
    )
    insufficient_span = bool(raw_span < config.minimum_reliability_p05_p95_span)
    degenerate = bool(saturated_uniform or insufficient_span)

    return {
        "source_posterior_median": source_median,
        "entropy_median": entropy_median,
        "raw_reliability_p05": float(raw_p05),
        "raw_reliability_p95": float(raw_p95),
        "raw_reliability_p05_p95_span": raw_span,
        "selected_ess_ratio": ess_ratio,
        "saturated_uniform_reliability": saturated_uniform,
        "insufficient_reliability_span": insufficient_span,
        "reliability_degenerate": degenerate,
    }

In [ ]:

# Cell 12 — Metrics

def multiclass_brier_score(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
) -> float:
    label_to_position = {
        int(label): position for position, label in enumerate(class_labels)
    }
    one_hot = np.zeros_like(probabilities, dtype=np.float64)
    positions = np.array([label_to_position[int(label)] for label in y_true], dtype=int)
    one_hot[np.arange(len(y_true)), positions] = 1.0
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))


def expected_calibration_error(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
    n_bins: int = 15,
) -> float:
    predicted_positions = probabilities.argmax(axis=1)
    predicted_labels = class_labels[predicted_positions]
    confidence = probabilities.max(axis=1)
    correct = (predicted_labels == y_true).astype(np.float64)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for bin_index in range(n_bins):
        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]
        if bin_index == 0:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence > lower) & (confidence <= upper)
        if not mask.any():
            continue
        bin_accuracy = correct[mask].mean()
        bin_confidence = confidence[mask].mean()
        ece += mask.mean() * abs(bin_accuracy - bin_confidence)
    return float(ece)


def evaluate_probabilities(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
    ece_bins: int,
) -> Dict[str, float]:
    predicted_labels = class_labels[probabilities.argmax(axis=1)]
    return {
        "macro_f1": float(
            f1_score(y_true, predicted_labels, labels=class_labels, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(y_true, predicted_labels)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted_labels)),
        "nll": float(log_loss(y_true, probabilities, labels=class_labels)),
        "ece": expected_calibration_error(
            y_true, probabilities, class_labels, n_bins=ece_bins
        ),
        "brier": multiclass_brier_score(y_true, probabilities, class_labels),
    }


def ncm_probabilities(
    X: np.ndarray,
    means: np.ndarray,
) -> np.ndarray:
    squared_distances = np.sum((X[:, None, :] - means[None, :, :]) ** 2, axis=2)
    logits = -squared_distances
    logits -= logits.max(axis=1, keepdims=True)
    probabilities = np.exp(logits)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return probabilities

In [ ]:

# Cell 13 — Classifier fitting and frozen selection rules

@dataclass
class CandidateModel:
    method: str
    C: float
    lambda_syn: Optional[float]
    gamma: Optional[float]
    validation_metrics: Dict[str, float]
    model: Any
    synthetic_weight_summary: Dict[str, float]


def fit_logistic_regression(
    X_real: np.ndarray,
    y_real: np.ndarray,
    C: float,
    seed: int,
    max_iter: int,
    X_syn: Optional[np.ndarray] = None,
    y_syn: Optional[np.ndarray] = None,
    synthetic_weights: Optional[np.ndarray] = None,
) -> LogisticRegression:
    if X_syn is None:
        X_train = X_real
        y_train = y_real
        sample_weight = np.ones(len(y_real), dtype=np.float64)
    else:
        if y_syn is None:
            raise ValueError("y_syn is required when X_syn is provided.")
        X_train = np.vstack([X_real, X_syn])
        y_train = np.concatenate([y_real, y_syn])
        if synthetic_weights is None:
            synthetic_weights = np.ones(len(y_syn), dtype=np.float64)
        sample_weight = np.concatenate([
            np.ones(len(y_real), dtype=np.float64),
            np.asarray(synthetic_weights, dtype=np.float64),
        ])

    model = LogisticRegression(
        C=float(C),
        solver="lbfgs",
        max_iter=int(max_iter),
        random_state=int(seed),
    )
    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def aligned_predict_proba(
    model: Any,
    X: np.ndarray,
    class_labels: np.ndarray,
) -> np.ndarray:
    raw = model.predict_proba(X)
    aligned = np.zeros((len(X), len(class_labels)), dtype=np.float64)
    position = {int(label): idx for idx, label in enumerate(class_labels)}
    for model_position, label in enumerate(model.classes_):
        aligned[:, position[int(label)]] = raw[:, model_position]
    aligned = np.clip(aligned, 1e-15, 1.0)
    aligned /= aligned.sum(axis=1, keepdims=True)
    return aligned


def candidate_sort_key(row: CandidateModel) -> Tuple[float, float, float]:
    lambda_value = float("inf") if row.lambda_syn is None else float(row.lambda_syn)
    gamma_value = float("inf") if row.gamma is None else float(row.gamma)
    return (lambda_value, gamma_value, float(row.C))


def select_f1_only(candidates: Sequence[CandidateModel]) -> CandidateModel:
    best_f1 = max(candidate.validation_metrics["macro_f1"] for candidate in candidates)
    stage = [
        candidate
        for candidate in candidates
        if candidate.validation_metrics["macro_f1"] >= best_f1 - 1e-12
    ]
    return sorted(stage, key=candidate_sort_key)[0]


def select_multiobjective(
    candidates: Sequence[CandidateModel],
    config: CAPFGConfig,
) -> CandidateModel:
    best_f1 = max(candidate.validation_metrics["macro_f1"] for candidate in candidates)
    stage_f1 = [
        candidate
        for candidate in candidates
        if candidate.validation_metrics["macro_f1"] >= best_f1 - config.f1_tolerance
    ]

    best_nll = min(candidate.validation_metrics["nll"] for candidate in stage_f1)
    stage_nll = [
        candidate
        for candidate in stage_f1
        if candidate.validation_metrics["nll"] <= best_nll + config.nll_tolerance
    ]

    best_ece = min(candidate.validation_metrics["ece"] for candidate in stage_nll)
    stage_ece = [
        candidate
        for candidate in stage_nll
        if candidate.validation_metrics["ece"] <= best_ece + config.ece_tolerance
    ]

    return sorted(stage_ece, key=candidate_sort_key)[0]


def weight_summary(weights: Optional[np.ndarray]) -> Dict[str, float]:
    if weights is None:
        return {
            "synthetic_weight_sum": 0.0,
            "synthetic_weight_min": 0.0,
            "synthetic_weight_max": 0.0,
            "ess": 0.0,
            "ess_ratio": 0.0,
            "gini": 0.0,
            "top_1pct_budget": 0.0,
            "top_5pct_budget": 0.0,
            "top_10pct_budget": 0.0,
        }
    weights = np.asarray(weights, dtype=np.float64)
    ess = effective_sample_size(weights)
    return {
        "synthetic_weight_sum": float(weights.sum()),
        "synthetic_weight_min": float(weights.min()),
        "synthetic_weight_max": float(weights.max()),
        "ess": ess,
        "ess_ratio": float(ess / len(weights)),
        "gini": gini_coefficient(weights),
        "top_1pct_budget": top_budget_fraction(weights, 0.01),
        "top_5pct_budget": top_budget_fraction(weights, 0.05),
        "top_10pct_budget": top_budget_fraction(weights, 0.10),
    }


def evaluate_candidate(
    candidate: CandidateModel,
    X: np.ndarray,
    y: np.ndarray,
    class_labels: np.ndarray,
    config: CAPFGConfig,
) -> Tuple[np.ndarray, Dict[str, float]]:
    probabilities = aligned_predict_proba(candidate.model, X, class_labels)
    metrics = evaluate_probabilities(y, probabilities, class_labels, config.ece_bins)
    return probabilities, metrics


In [ ]:

# Cell 14 — Frozen confirmatory task engine

def make_task_key(dataset: str, k: int, seed: int) -> str:
    return f"{dataset}__k{k}__seed{seed}"


def candidate_record(
    task_key: str,
    candidate: CandidateModel,
    selected_by: str = "",
) -> Dict[str, Any]:
    record = {
        "task_key": task_key,
        "method": candidate.method,
        "C": candidate.C,
        "lambda_syn": candidate.lambda_syn,
        "gamma": candidate.gamma,
        "selected_by": selected_by,
    }
    record.update({f"validation_{k}": v for k, v in candidate.validation_metrics.items()})
    record.update(candidate.synthetic_weight_summary)
    return record


def prediction_frame(
    task_key: str,
    dataset: str,
    k: int,
    seed: int,
    method: str,
    split_name: str,
    sample_keys: Sequence[str],
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
) -> pd.DataFrame:
    predicted_labels = class_labels[probabilities.argmax(axis=1)]
    return pd.DataFrame({
        "task_key": task_key,
        "dataset": dataset,
        "K": k,
        "seed": seed,
        "method": method,
        "split": split_name,
        "sample_key": list(sample_keys),
        "true_label": y_true.astype(int),
        "predicted_label": predicted_labels.astype(int),
        "probability_vector": [
            json.dumps(row.tolist(), separators=(",", ":")) for row in probabilities
        ],
    })


def run_confirmatory_task(
    bundle: DatasetBundle,
    k: int,
    seed: int,
    config: CAPFGConfig,
    include_ablations: bool = True,
) -> Dict[str, Any]:
    task_key = make_task_key(bundle.dataset, k, seed)
    meta = bundle.metadata
    split = build_task_split(bundle, k, seed)

    support_idx = split.support_indices
    validation_idx = split.validation_indices
    test_idx = split.test_indices

    X_support_raw = bundle.features[support_idx]
    X_validation_raw = bundle.features[validation_idx]
    X_test_raw = bundle.features[test_idx]

    y_support = meta.loc[support_idx, "label_canonical"].to_numpy(dtype=int)
    y_validation = meta.loc[validation_idx, "label_canonical"].to_numpy(dtype=int)
    y_test = meta.loc[test_idx, "label_canonical"].to_numpy(dtype=int)

    # Support-only representation.
    preprocessor = fit_support_preprocessor(
        X_support_raw,
        config,
        seed=derive_seed(seed, "pca"),
    )
    X_support = transform_with_support_preprocessor(preprocessor, X_support_raw)
    X_validation = transform_with_support_preprocessor(preprocessor, X_validation_raw)
    X_test = transform_with_support_preprocessor(preprocessor, X_test_raw)

    gaussian_model = estimate_gaussian_model(
        X_support,
        y_support,
        split.class_labels,
        pooled_weight=config.pooled_covariance_weight,
        jitter=config.gaussian_jitter,
    )

    # Frozen main generator: no hardness scaling.
    unit_scales = np.ones(len(split.class_labels), dtype=np.float64)
    main_generation_covariances = construct_generation_covariances(
        gaussian_model.base_covariances,
        unit_scales,
        config.gaussian_jitter,
    )

    samples_per_class = config.synthetic_multiplier * k
    synthetic_main = generate_antithetic_features(
        gaussian_model,
        main_generation_covariances,
        samples_per_class,
        seed,
        geometry_name="frozen_main_unscaled",
        use_antithetic=config.use_antithetic_sampling,
    )

    # Controlled ablation geometry and reliability.
    hardness = None
    scales = None
    scaled_covariances = None
    synthetic_scaled = None
    temperature_calibration = None
    reliability = None
    reliability_fixed = None

    if include_ablations:
        hardness = compute_leave_one_out_hardness(
            X_support,
            y_support,
            gaussian_model,
            config,
        )
        scales = covariance_scales_from_hardness(
            hardness.final_hardness,
            config.fixed_min_covariance_scale,
        )
        scaled_covariances = construct_generation_covariances(
            gaussian_model.base_covariances,
            scales,
            config.gaussian_jitter,
        )
        synthetic_scaled = generate_antithetic_features(
            gaussian_model,
            scaled_covariances,
            samples_per_class,
            seed,
            geometry_name="cf_scaled_ablation",
            use_antithetic=config.use_antithetic_sampling,
        )

        temperature_calibration = calibrate_shared_temperature(
            X_support,
            y_support,
            gaussian_model,
            config,
        )
        reliability = compute_synthetic_reliability_v2(
            synthetic_scaled,
            gaussian_model,
            scaled_covariances,
            temperature_calibration,
            config,
        )
        reliability_fixed = powered_reliability(
            reliability.raw_reliability,
            config.reliability_gamma_fixed,
            config.reliability_minimum,
        )

    selected: Dict[str, CandidateModel] = {}
    all_candidates: List[CandidateModel] = []
    direct_results: Dict[str, Dict[str, Any]] = {}

    # 1) NCM-Real
    ncm_val_prob = ncm_probabilities(X_validation, gaussian_model.means)
    ncm_test_prob = ncm_probabilities(X_test, gaussian_model.means)
    direct_results["NCM-Real"] = {
        "validation_prob": ncm_val_prob,
        "test_prob": ncm_test_prob,
        "validation_metrics": evaluate_probabilities(
            y_validation, ncm_val_prob, split.class_labels, config.ece_bins
        ),
        "test_metrics": evaluate_probabilities(
            y_test, ncm_test_prob, split.class_labels, config.ece_bins
        ),
    }

    # 2) OAS-Gaussian: pure class-wise OAS covariance baseline.
    oas_val_prob = gaussian_posterior(
        X_validation,
        gaussian_model.means,
        gaussian_model.class_covariances,
    )
    oas_test_prob = gaussian_posterior(
        X_test,
        gaussian_model.means,
        gaussian_model.class_covariances,
    )
    direct_results["OAS-Gaussian"] = {
        "validation_prob": oas_val_prob,
        "test_prob": oas_test_prob,
        "validation_metrics": evaluate_probabilities(
            y_validation, oas_val_prob, split.class_labels, config.ece_bins
        ),
        "test_metrics": evaluate_probabilities(
            y_test, oas_test_prob, split.class_labels, config.ece_bins
        ),
    }

    # 3) Shrinkage-LDA
    lda_model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    lda_model.fit(X_support, y_support)
    lda_val_prob = aligned_predict_proba(lda_model, X_validation, split.class_labels)
    lda_test_prob = aligned_predict_proba(lda_model, X_test, split.class_labels)
    direct_results["Shrinkage-LDA"] = {
        "validation_prob": lda_val_prob,
        "test_prob": lda_test_prob,
        "validation_metrics": evaluate_probabilities(
            y_validation, lda_val_prob, split.class_labels, config.ece_bins
        ),
        "test_metrics": evaluate_probabilities(
            y_test, lda_test_prob, split.class_labels, config.ece_bins
        ),
    }

    # 4) LR-Real: C selected only by validation Macro-F1.
    lr_candidates = []
    for C in config.logreg_c_grid:
        model = fit_logistic_regression(
            X_support, y_support,
            C=C,
            seed=derive_seed(seed, f"lr_real::{C}"),
            max_iter=config.max_iter_logreg,
        )
        val_prob = aligned_predict_proba(model, X_validation, split.class_labels)
        candidate = CandidateModel(
            method="LR-Real",
            C=C,
            lambda_syn=None,
            gamma=None,
            validation_metrics=evaluate_probabilities(
                y_validation, val_prob, split.class_labels, config.ece_bins
            ),
            model=model,
            synthetic_weight_summary=weight_summary(None),
        )
        lr_candidates.append(candidate)
        all_candidates.append(candidate)
    selected["LR-Real"] = select_f1_only(lr_candidates)

    # 5) PFG-UnweightedLR: generated features have unit weight.
    unweighted_candidates = []
    unit_weights = np.ones(len(synthetic_main.features), dtype=np.float64)
    for C in config.logreg_c_grid:
        model = fit_logistic_regression(
            X_support, y_support,
            C=C,
            seed=derive_seed(seed, f"pfg_unweighted::{C}"),
            max_iter=config.max_iter_logreg,
            X_syn=synthetic_main.features,
            y_syn=synthetic_main.labels,
            synthetic_weights=unit_weights,
        )
        val_prob = aligned_predict_proba(model, X_validation, split.class_labels)
        candidate = CandidateModel(
            method="PFG-UnweightedLR",
            C=C,
            lambda_syn=None,
            gamma=None,
            validation_metrics=evaluate_probabilities(
                y_validation, val_prob, split.class_labels, config.ece_bins
            ),
            model=model,
            synthetic_weight_summary=weight_summary(unit_weights),
        )
        unweighted_candidates.append(candidate)
        all_candidates.append(candidate)
    selected["PFG-UnweightedLR"] = select_f1_only(unweighted_candidates)

    # 6/7) One frozen normalized-budget candidate pool, two selectors.
    main_budget_candidates = []
    uniform_reliability = np.ones(len(synthetic_main.features), dtype=np.float64)
    for lambda_syn in config.synthetic_total_weight_grid:
        weights = normalize_synthetic_weights(
            uniform_reliability,
            lambda_syn,
            len(X_support),
            config.numerical_tolerance,
        )
        for C in config.logreg_c_grid:
            model = fit_logistic_regression(
                X_support, y_support,
                C=C,
                seed=derive_seed(seed, f"pfg_nw::{lambda_syn}::{C}"),
                max_iter=config.max_iter_logreg,
                X_syn=synthetic_main.features,
                y_syn=synthetic_main.labels,
                synthetic_weights=weights,
            )
            val_prob = aligned_predict_proba(model, X_validation, split.class_labels)
            candidate = CandidateModel(
                method="PFG-NW",
                C=C,
                lambda_syn=lambda_syn,
                gamma=None,
                validation_metrics=evaluate_probabilities(
                    y_validation, val_prob, split.class_labels, config.ece_bins
                ),
                model=model,
                synthetic_weight_summary=weight_summary(weights),
            )
            main_budget_candidates.append(candidate)
            all_candidates.append(candidate)

    selected["PFG-NW-F1Only"] = select_f1_only(main_budget_candidates)
    selected["CA-PFG-MO"] = select_multiobjective(main_budget_candidates, config)

    # Controlled ablations.
    selected_reliability_weights = None
    reliability_degeneracy = None

    if include_ablations:
        # 8) Class-hardness scalar covariance scaling + normalized budget + MO.
        cf_candidates = []
        uniform_scaled = np.ones(len(synthetic_scaled.features), dtype=np.float64)
        for lambda_syn in config.synthetic_total_weight_grid:
            weights = normalize_synthetic_weights(
                uniform_scaled,
                lambda_syn,
                len(X_support),
                config.numerical_tolerance,
            )
            for C in config.logreg_c_grid:
                model = fit_logistic_regression(
                    X_support, y_support,
                    C=C,
                    seed=derive_seed(seed, f"cf_scale::{lambda_syn}::{C}"),
                    max_iter=config.max_iter_logreg,
                    X_syn=synthetic_scaled.features,
                    y_syn=synthetic_scaled.labels,
                    synthetic_weights=weights,
                )
                val_prob = aligned_predict_proba(model, X_validation, split.class_labels)
                candidate = CandidateModel(
                    method="CF-Scale-NWLR-MO",
                    C=C,
                    lambda_syn=lambda_syn,
                    gamma=None,
                    validation_metrics=evaluate_probabilities(
                        y_validation, val_prob, split.class_labels, config.ece_bins
                    ),
                    model=model,
                    synthetic_weight_summary=weight_summary(weights),
                )
                cf_candidates.append(candidate)
                all_candidates.append(candidate)
        selected["CF-Scale-NWLR-MO"] = select_multiobjective(cf_candidates, config)

        # 9/10) Reliability-v2 with frozen gamma=0.5.
        reliability_candidates = []
        for lambda_syn in config.synthetic_total_weight_grid:
            weights = normalize_class_balanced_synthetic_weights(
                reliability_fixed,
                synthetic_scaled.source_positions,
                lambda_syn,
                len(X_support),
                len(split.class_labels),
                config.numerical_tolerance,
            )
            for C in config.logreg_c_grid:
                model = fit_logistic_regression(
                    X_support, y_support,
                    C=C,
                    seed=derive_seed(
                        seed,
                        f"reliability_v2::{config.reliability_gamma_fixed}::{lambda_syn}::{C}"
                    ),
                    max_iter=config.max_iter_logreg,
                    X_syn=synthetic_scaled.features,
                    y_syn=synthetic_scaled.labels,
                    synthetic_weights=weights,
                )
                val_prob = aligned_predict_proba(model, X_validation, split.class_labels)
                candidate = CandidateModel(
                    method="Reliability-v2",
                    C=C,
                    lambda_syn=lambda_syn,
                    gamma=config.reliability_gamma_fixed,
                    validation_metrics=evaluate_probabilities(
                        y_validation, val_prob, split.class_labels, config.ece_bins
                    ),
                    model=model,
                    synthetic_weight_summary=weight_summary(weights),
                )
                reliability_candidates.append(candidate)
                all_candidates.append(candidate)

        selected["Reliability-v2-F1Only"] = select_f1_only(reliability_candidates)
        selected["Reliability-v2-MO"] = select_multiobjective(reliability_candidates, config)

        selected_rel = selected["Reliability-v2-MO"]
        selected_reliability_weights = normalize_class_balanced_synthetic_weights(
            reliability_fixed,
            synthetic_scaled.source_positions,
            float(selected_rel.lambda_syn),
            len(X_support),
            len(split.class_labels),
            config.numerical_tolerance,
        )
        reliability_degeneracy = reliability_degeneracy_summary(
            reliability,
            selected_reliability_weights,
            config,
        )

    # Test evaluation only after all validation-based selections are complete.
    selected_evaluations: Dict[str, Dict[str, Any]] = {}
    for method_name, candidate in selected.items():
        val_prob, val_metrics = evaluate_candidate(
            candidate, X_validation, y_validation, split.class_labels, config
        )
        test_prob, test_metrics = evaluate_candidate(
            candidate, X_test, y_test, split.class_labels, config
        )
        selected_evaluations[method_name] = {
            "validation_prob": val_prob,
            "test_prob": test_prob,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics,
            "candidate": candidate,
        }

    # Main result rows.
    task_result_rows = []

    for method_name, result in direct_results.items():
        row = {
            "task_key": task_key,
            "dataset": bundle.dataset,
            "K": k,
            "seed": seed,
            "method": method_name,
            "method_group": "main_comparison",
            "selected_C": None,
            "selected_lambda_syn": None,
            "selected_gamma": None,
            "pca_components": preprocessor.n_components,
            "config_hash": CONFIG_HASH,
        }
        row.update({f"validation_{key}": value for key, value in result["validation_metrics"].items()})
        row.update({f"test_{key}": value for key, value in result["test_metrics"].items()})
        row.update(weight_summary(None))
        task_result_rows.append(row)

    for method_name, result in selected_evaluations.items():
        candidate = result["candidate"]
        group = (
            "controlled_ablation"
            if method_name in {
                "CF-Scale-NWLR-MO",
                "Reliability-v2-F1Only",
                "Reliability-v2-MO",
            }
            else "main_comparison"
        )
        row = {
            "task_key": task_key,
            "dataset": bundle.dataset,
            "K": k,
            "seed": seed,
            "method": method_name,
            "method_group": group,
            "selected_C": candidate.C,
            "selected_lambda_syn": candidate.lambda_syn,
            "selected_gamma": candidate.gamma,
            "pca_components": preprocessor.n_components,
            "config_hash": CONFIG_HASH,
        }
        row.update({f"validation_{key}": value for key, value in result["validation_metrics"].items()})
        row.update({f"test_{key}": value for key, value in result["test_metrics"].items()})
        row.update(candidate.synthetic_weight_summary)
        task_result_rows.append(row)

    # Candidate rows and selector annotation.
    selected_identity = {}
    for name, candidate in selected.items():
        if name.endswith("F1Only") or name in {"LR-Real", "PFG-UnweightedLR"}:
            selector = "f1_only"
        else:
            selector = "multiobjective"
        selected_identity[id(candidate)] = name + "::" + selector

    candidate_rows = [
        candidate_record(
            task_key,
            candidate,
            selected_identity.get(id(candidate), ""),
        )
        for candidate in all_candidates
    ]

    # Predictions: validation and test for every reported method.
    prediction_frames = []

    reported_probabilities = {}
    for method_name, result in direct_results.items():
        reported_probabilities[method_name] = result
    for method_name, result in selected_evaluations.items():
        reported_probabilities[method_name] = result

    for method_name, result in reported_probabilities.items():
        for split_name, indices, y_true, prob_key in (
            ("validation", validation_idx, y_validation, "validation_prob"),
            ("test", test_idx, y_test, "test_prob"),
        ):
            prediction_frames.append(
                prediction_frame(
                    task_key=task_key,
                    dataset=bundle.dataset,
                    k=k,
                    seed=seed,
                    method=method_name,
                    split_name=split_name,
                    sample_keys=meta.loc[indices, "sample_key"].tolist(),
                    y_true=y_true,
                    probabilities=result[prob_key],
                    class_labels=split.class_labels,
                )
            )

    # Synthetic diagnostics focus on the frozen main generator.
    main_diag = pd.DataFrame({
        "task_key": task_key,
        "dataset": bundle.dataset,
        "K": k,
        "seed": seed,
        "synthetic_id": np.arange(len(synthetic_main.features), dtype=int),
        "source_label": synthetic_main.labels.astype(int),
        "source_position": synthetic_main.source_positions.astype(int),
        "pair_id": synthetic_main.pair_ids.astype(int),
        "geometry": "frozen_main_unscaled",
    })

    selected_main = selected["CA-PFG-MO"]
    main_weights = normalize_synthetic_weights(
        uniform_reliability,
        float(selected_main.lambda_syn),
        len(X_support),
        config.numerical_tolerance,
    )
    main_diag["selected_ca_pfg_mo_weight"] = main_weights

    # Add reliability ablation diagnostics to a separate table.
    reliability_diag = pd.DataFrame()
    class_diag = pd.DataFrame()
    temperature_diag = pd.DataFrame()

    if include_ablations:
        reliability_diag = pd.DataFrame({
            "task_key": task_key,
            "dataset": bundle.dataset,
            "K": k,
            "seed": seed,
            "synthetic_id": np.arange(len(synthetic_scaled.features), dtype=int),
            "source_label": synthetic_scaled.labels.astype(int),
            "source_position": synthetic_scaled.source_positions.astype(int),
            "pair_id": synthetic_scaled.pair_ids.astype(int),
            "source_posterior": reliability.source_posterior,
            "normalized_entropy": reliability.normalized_entropy,
            "radial_centrality": reliability.radial_centrality,
            "raw_reliability": reliability.raw_reliability,
            "selected_reliability_weight": selected_reliability_weights,
            "source_disagreement": (
                reliability.predicted_positions != synthetic_scaled.source_positions
            ).astype(int),
            "shared_mahalanobis_to_source": reliability.shared_mahalanobis_to_source,
            "generation_mahalanobis_to_source": reliability.generation_mahalanobis_to_source,
        })

        class_rows = []
        for position, label in enumerate(split.class_labels):
            mask = synthetic_scaled.source_positions == position
            class_weights = selected_reliability_weights[mask]
            class_rows.append({
                "task_key": task_key,
                "dataset": bundle.dataset,
                "K": k,
                "seed": seed,
                "class_label": int(label),
                "class_hardness_raw": float(hardness.raw_hardness[position]),
                "class_hardness_final": float(hardness.final_hardness[position]),
                "covariance_scale": float(scales[position]),
                "oas_shrinkage": float(gaussian_model.oas_shrinkages[position]),
                "mean_source_posterior": float(reliability.source_posterior[mask].mean()),
                "mean_entropy": float(reliability.normalized_entropy[mask].mean()),
                "mean_radial_centrality": float(reliability.radial_centrality[mask].mean()),
                "mean_raw_reliability": float(reliability.raw_reliability[mask].mean()),
                "source_disagreement_rate": float(
                    (reliability.predicted_positions[mask] != position).mean()
                ),
                "within_class_ess_ratio": float(
                    effective_sample_size(class_weights) / len(class_weights)
                ),
            })
        class_diag = pd.DataFrame(class_rows)

        temperature_diag = temperature_calibration.candidate_table.copy()
        temperature_diag.insert(0, "task_key", task_key)
        temperature_diag.insert(1, "dataset", bundle.dataset)
        temperature_diag.insert(2, "K", k)
        temperature_diag.insert(3, "seed", seed)
        temperature_diag["selected"] = np.isclose(
            temperature_diag["temperature"],
            temperature_calibration.selected_temperature,
        )

    # Protocol audit.
    support_keys = set(meta.loc[support_idx, "sample_key"])
    validation_keys = set(meta.loc[validation_idx, "sample_key"])
    test_keys = set(meta.loc[test_idx, "sample_key"])

    main_weight_sum = float(main_weights.sum())
    expected_main_weight_sum = float(selected_main.lambda_syn) * len(X_support)

    protocol_checks = {
        "support_validation_disjoint": len(support_keys & validation_keys) == 0,
        "support_test_disjoint": len(support_keys & test_keys) == 0,
        "validation_test_disjoint": len(validation_keys & test_keys) == 0,
        "support_exact_k_per_class": bool(
            np.all(np.unique(y_support, return_counts=True)[1] == k)
        ),
        "pca_fitted_on_support_count": int(preprocessor.pca.n_samples_) == len(X_support),
        "gaussian_estimation_support_only": True,
        "synthetic_generation_support_only": True,
        "validation_for_selection_only": True,
        "test_evaluated_after_selection": True,
        "main_synthetic_budget_conserved": bool(
            abs(main_weight_sum - expected_main_weight_sum)
            <= config.numerical_tolerance
        ),
        "config_hash_match": CONFIG_HASH == stable_hash_config(CONFIG),
    }

    overall_pass = all(protocol_checks.values())

    protocol_row = {
        "task_key": task_key,
        "dataset": bundle.dataset,
        "K": k,
        "seed": seed,
        **protocol_checks,
        "overall_protocol_pass": overall_pass,
        "main_selected_lambda_syn": float(selected_main.lambda_syn),
        "main_selected_C": float(selected_main.C),
        "main_weight_sum": main_weight_sum,
        "main_expected_weight_sum": expected_main_weight_sum,
        "config_hash": CONFIG_HASH,
    }

    if include_ablations:
        protocol_row["temperature_calibration_support_only"] = True
        protocol_row["selected_temperature"] = float(
            temperature_calibration.selected_temperature
        )
        protocol_row.update(reliability_degeneracy)

    if not overall_pass:
        failed = [key for key, value in protocol_checks.items() if not value]
        raise AssertionError(f"{task_key}: protocol audit failed: {failed}")

    return {
        "task_results": pd.DataFrame(task_result_rows),
        "candidate_results": pd.DataFrame(candidate_rows),
        "predictions": pd.concat(prediction_frames, ignore_index=True),
        "main_synthetic_diagnostics": main_diag,
        "reliability_diagnostics": reliability_diag,
        "class_diagnostics": class_diag,
        "temperature_diagnostics": temperature_diag,
        "protocol_audit": pd.DataFrame([protocol_row]),
    }


In [ ]:

# Cell 15 — Incremental export helpers

def safe_write_table(frame: pd.DataFrame, parquet_path: Path) -> Path:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = parquet_path.with_suffix(".csv")
        print(f"[WARN] Parquet export failed ({exc}); writing CSV: {csv_path}")
        frame.to_csv(csv_path, index=False)
        return csv_path


def save_large_task_outputs(output: Dict[str, Any], task_key: str) -> Dict[str, Path]:
    saved = {}
    saved["predictions"] = safe_write_table(
        output["predictions"],
        OUTPUT_DIR / "predictions" / f"{task_key}.parquet",
    )
    saved["main_synthetic_diagnostics"] = safe_write_table(
        output["main_synthetic_diagnostics"],
        OUTPUT_DIR / "synthetic_diagnostics" / f"{task_key}__main.parquet",
    )
    if len(output["reliability_diagnostics"]):
        saved["reliability_diagnostics"] = safe_write_table(
            output["reliability_diagnostics"],
            OUTPUT_DIR / "synthetic_diagnostics" / f"{task_key}__reliability_v2.parquet",
        )
    return saved


def collect_small_table(
    small_outputs: Sequence[Dict[str, pd.DataFrame]],
    key: str,
) -> pd.DataFrame:
    frames = [entry[key] for entry in small_outputs if key in entry and len(entry[key])]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def save_small_output_bundle(
    small_outputs: Sequence[Dict[str, pd.DataFrame]],
    prefix: str,
) -> Dict[str, Path]:
    mappings = {
        "task_results": f"{prefix}_task_results.csv",
        "candidate_results": f"{prefix}_candidate_results.csv",
        "class_diagnostics": f"{prefix}_class_diagnostics.csv",
        "temperature_diagnostics": f"{prefix}_temperature_diagnostics.csv",
        "protocol_audit": f"{prefix}_protocol_audit.csv",
    }
    saved = {}
    for key, filename in mappings.items():
        frame = collect_small_table(small_outputs, key)
        path = OUTPUT_DIR / filename
        frame.to_csv(path, index=False)
        saved[key] = path
    return saved


def small_view(output: Dict[str, Any]) -> Dict[str, pd.DataFrame]:
    return {
        "task_results": output["task_results"],
        "candidate_results": output["candidate_results"],
        "class_diagnostics": output["class_diagnostics"],
        "temperature_diagnostics": output["temperature_diagnostics"],
        "protocol_audit": output["protocol_audit"],
    }


def create_output_zip(zip_name: str = "CAPFG10_confirmatory_package.zip") -> Path:
    archive_base = OUTPUT_DIR.parent / Path(zip_name).stem
    archive_path = Path(
        shutil.make_archive(
            str(archive_base.resolve()),
            "zip",
            root_dir=OUTPUT_DIR,
        )
    )
    return archive_path



## Technical smoke test

Run exactly one task first:

- PlantDoc
- \(K=5\)
- seed 137

The smoke run must satisfy:

```text
overall_protocol_pass = True
```

Only then enable the 60-task confirmatory run.


In [ ]:

# Cell 17 — Run technical smoke test

SMOKE_OUTPUT = run_confirmatory_task(
    BUNDLES[CONFIG.smoke_dataset],
    k=CONFIG.smoke_k,
    seed=CONFIG.smoke_seed,
    config=CONFIG,
    include_ablations=INCLUDE_CONTROLLED_ABLATIONS,
)

SMOKE_TASK_KEY = make_task_key(
    CONFIG.smoke_dataset,
    CONFIG.smoke_k,
    CONFIG.smoke_seed,
)

save_large_task_outputs(SMOKE_OUTPUT, SMOKE_TASK_KEY)
save_small_output_bundle([small_view(SMOKE_OUTPUT)], prefix="smoke")

print("Smoke test completed:", SMOKE_TASK_KEY)
display(
    SMOKE_OUTPUT["task_results"]
    .sort_values(["method_group", "method"])
    .reset_index(drop=True)
)
display(SMOKE_OUTPUT["protocol_audit"].T)

SMOKE_READY_FOR_FULL = bool(
    SMOKE_OUTPUT["protocol_audit"].iloc[0]["overall_protocol_pass"]
)
print("SMOKE_READY_FOR_FULL:", SMOKE_READY_FOR_FULL)


In [ ]:

# Cell 18 — Smoke comparison diagnostics

smoke_results = SMOKE_OUTPUT["task_results"].copy()

display(
    smoke_results[
        [
            "method_group",
            "method",
            "selected_C",
            "selected_lambda_syn",
            "selected_gamma",
            "validation_macro_f1",
            "validation_nll",
            "validation_ece",
            "test_macro_f1",
            "test_nll",
            "test_ece",
            "test_brier",
        ]
    ]
    .sort_values(["method_group", "method"])
    .reset_index(drop=True)
)

main_methods = smoke_results.loc[
    smoke_results["method_group"] == "main_comparison"
].copy()

fig = plt.figure(figsize=(8, 5))
plt.scatter(main_methods["test_macro_f1"], main_methods["test_ece"])
for _, row in main_methods.iterrows():
    plt.annotate(row["method"], (row["test_macro_f1"], row["test_ece"]))
plt.xlabel("Test Macro-F1")
plt.ylabel("Test ECE")
plt.title("Smoke task: discrimination–calibration trade-off")
plt.tight_layout()
plt.show()



## Full frozen confirmatory evaluation

After reviewing the smoke output, set:

```python
RUN_FULL_CONFIRMATORY = True
```

in Cell 3 and rerun the notebook.

Full task count:

\[
2 \text{ datasets}
\times
3 K\text{-levels}
\times
10 \text{ seeds}
=
60 \text{ tasks}.
\]

Large prediction and synthetic-diagnostic tables are written **per task** to disk to avoid holding the whole experiment in RAM.


In [ ]:

# Cell 20 — Run 60 frozen confirmatory tasks

CONFIRMATORY_SMALL_OUTPUTS: List[Dict[str, pd.DataFrame]] = []

if RUN_FULL_CONFIRMATORY:
    if not SMOKE_READY_FOR_FULL:
        raise RuntimeError(
            "Full confirmatory run is blocked because the smoke protocol audit failed."
        )

    expected_task_count = (
        len(CONFIG.confirmatory_datasets)
        * len(CONFIG.confirmatory_k_values)
        * len(CONFIG.confirmatory_seeds)
    )
    completed = 0

    for dataset_name in CONFIG.confirmatory_datasets:
        bundle = BUNDLES[dataset_name]

        for k in CONFIG.confirmatory_k_values:
            for seed in CONFIG.confirmatory_seeds:
                task_key = make_task_key(dataset_name, k, seed)

                if (
                    dataset_name == CONFIG.smoke_dataset
                    and k == CONFIG.smoke_k
                    and seed == CONFIG.smoke_seed
                ):
                    output = SMOKE_OUTPUT
                    print(f"[REUSE] {task_key}")
                else:
                    print(f"[RUN]   {task_key}")
                    output = run_confirmatory_task(
                        bundle,
                        k=k,
                        seed=seed,
                        config=CONFIG,
                        include_ablations=INCLUDE_CONTROLLED_ABLATIONS,
                    )

                save_large_task_outputs(output, task_key)
                CONFIRMATORY_SMALL_OUTPUTS.append(small_view(output))

                completed += 1
                print(f"Completed: {completed} / {expected_task_count}")

    save_small_output_bundle(
        CONFIRMATORY_SMALL_OUTPUTS,
        prefix="confirmatory",
    )

    print(f"Confirmatory evaluation completed: {completed}/{expected_task_count}")
else:
    print(
        "Full confirmatory run is disabled. "
        "Inspect the smoke result, then set RUN_FULL_CONFIRMATORY=True in Cell 3."
    )


In [ ]:

# Cell 21 — Confirmatory completion and protocol audit

if RUN_FULL_CONFIRMATORY:
    confirmatory_results = collect_small_table(
        CONFIRMATORY_SMALL_OUTPUTS,
        "task_results",
    )
    confirmatory_audit = collect_small_table(
        CONFIRMATORY_SMALL_OUTPUTS,
        "protocol_audit",
    )

    expected_task_count = (
        len(CONFIG.confirmatory_datasets)
        * len(CONFIG.confirmatory_k_values)
        * len(CONFIG.confirmatory_seeds)
    )

    unique_tasks = confirmatory_audit["task_key"].nunique()
    all_pass = bool(confirmatory_audit["overall_protocol_pass"].all())

    completion_report = {
        "expected_tasks": expected_task_count,
        "completed_unique_tasks": int(unique_tasks),
        "all_protocol_pass": all_pass,
        "config_hash": CONFIG_HASH,
    }

    write_json(completion_report, OUTPUT_DIR / "confirmatory_completion_report.json")
    print(json.dumps(completion_report, indent=2))

    if unique_tasks != expected_task_count:
        raise AssertionError(
            f"Expected {expected_task_count} tasks but found {unique_tasks}."
        )
    if not all_pass:
        failed = confirmatory_audit.loc[
            ~confirmatory_audit["overall_protocol_pass"],
            "task_key",
        ].tolist()
        raise AssertionError(f"Protocol audit failed for tasks: {failed}")

    print("Frozen confirmatory protocol audit: PASSED")


In [ ]:

# Cell 22 — Aggregate result summary

if RUN_FULL_CONFIRMATORY:
    metric_columns = [
        "test_macro_f1",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_nll",
        "test_ece",
        "test_brier",
    ]

    summary = (
        confirmatory_results
        .groupby(["method_group", "dataset", "K", "method"])[metric_columns]
        .agg(["mean", "std"])
        .reset_index()
    )
    summary.to_csv(OUTPUT_DIR / "confirmatory_summary_mean_std.csv", index=False)
    display(summary)

    main_summary = summary.loc[
        summary["method_group"] == "main_comparison"
    ].copy()
    display(main_summary)


In [ ]:

# Cell 23 — Paired comparisons central to the paper

def paired_comparison_table(
    results: pd.DataFrame,
    method: str,
    baseline: str,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows = []

    for (dataset, k), subset in results.groupby(["dataset", "K"]):
        left = subset.loc[subset["method"] == method].set_index("seed")
        right = subset.loc[subset["method"] == baseline].set_index("seed")
        common = left.index.intersection(right.index)

        for metric in metrics:
            delta = left.loc[common, metric] - right.loc[common, metric]
            higher_is_better = metric in {
                "test_macro_f1",
                "test_accuracy",
                "test_balanced_accuracy",
            }

            if higher_is_better:
                wins = int((delta > 1e-12).sum())
                losses = int((delta < -1e-12).sum())
            else:
                wins = int((delta < -1e-12).sum())
                losses = int((delta > 1e-12).sum())

            rows.append({
                "dataset": dataset,
                "K": int(k),
                "method": method,
                "baseline": baseline,
                "metric": metric,
                "mean_signed_delta": float(delta.mean()),
                "std_signed_delta": float(delta.std(ddof=1)),
                "wins": wins,
                "ties": int((np.abs(delta) <= 1e-12).sum()),
                "losses": losses,
                "n_pairs": int(len(common)),
            })

    return pd.DataFrame(rows)


if RUN_FULL_CONFIRMATORY:
    core_metrics = [
        "test_macro_f1",
        "test_nll",
        "test_ece",
        "test_brier",
    ]

    comparison_specs = [
        ("PFG-NW-F1Only", "PFG-UnweightedLR"),
        ("CA-PFG-MO", "PFG-NW-F1Only"),
        ("CA-PFG-MO", "LR-Real"),
        ("CA-PFG-MO", "OAS-Gaussian"),
        ("CA-PFG-MO", "Shrinkage-LDA"),
    ]

    if INCLUDE_CONTROLLED_ABLATIONS:
        comparison_specs.extend([
            ("CA-PFG-MO", "CF-Scale-NWLR-MO"),
            ("Reliability-v2-MO", "CA-PFG-MO"),
            ("Reliability-v2-MO", "Reliability-v2-F1Only"),
        ])

    paired_tables = [
        paired_comparison_table(
            confirmatory_results,
            method,
            baseline,
            core_metrics,
        )
        for method, baseline in comparison_specs
    ]

    paired_results = pd.concat(paired_tables, ignore_index=True)
    paired_results.to_csv(
        OUTPUT_DIR / "confirmatory_paired_comparisons.csv",
        index=False,
    )
    display(paired_results)


In [ ]:

# Cell 24 — Final package export

archive_path = create_output_zip("CAPFG10_confirmatory_package.zip")

print("Output package:", archive_path)
print("Top-level output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)



## Frozen interpretation rules

The confirmatory run is intended to evaluate the following pre-specified claims.

### Claim A — normalized synthetic influence budget

Compare:

```text
PFG-NW-F1Only
vs
PFG-UnweightedLR
```

Expected evidence: substantially improved NLL/ECE/Brier, with classification performance preserved or improved.

### Claim B — calibration-aware model selection

Compare:

```text
CA-PFG-MO
vs
PFG-NW-F1Only
```

Expected evidence: better NLL/ECE/Brier with only negligible Macro-F1 loss.

### Claim C — competitive classification with improved calibration

Compare:

```text
CA-PFG-MO
vs
LR-Real
```

The method does not have to dominate LR-Real in Macro-F1. The intended contribution is competitive discrimination together with more reliable predictive probabilities.

### Controlled ablations

The following are not part of the frozen main method:

- `CF-Scale-NWLR-MO`
- `Reliability-v2-F1Only`
- `Reliability-v2-MO`

They are retained only to document whether additional complexity produces consistent benefit under independent confirmatory seeds.
